# 4단계: 고정 파라미터 반복 Group CV 기반 최종 앙상블 비교

> 이전 실험 기록: 아래 코드와 출력은 TabICL을 포함한 이전 7:3 분할 실험이다. 현재 RF 베이스라인·튜닝·앙상블의 실행 순서는 [노트북 안내](README.md)를 따른다. 최신 결과와 수치를 섞지 않는다.

3단계에서 선택한 다섯 모델을 같은 원본 범주형 입력으로 다시 구성하고 다음 후보를 비교한다.

- 단일 모델 5개
- 동일 가중 Soft Voting
- Brier를 직접 줄이는 25회 Greedy Ensemble Selection(GES)
- OOF 확률을 입력으로 받는 규제 Logistic Stacking

3단계에서 선택한 최적 파라미터를 고정하고 5-Fold Group CV를 시드 3개로 반복한다. 각 바깥 Fold의 Train 안에서 3-Fold OOF 확률을 만든 뒤 GES 가중치와 Stacking을 학습하므로 앙상블 결합에 사용한 행과 바깥 평가 행이 겹치지 않는다. 3단계 파라미터는 전체 Train에서 이미 선택됐으므로 이 결과는 전체 튜닝 절차의 독립적인 nested CV 추정치가 아니라, 고정된 후보들의 조건부 앙상블 비교 결과다. 분류 임계값은 0.5로 고정한다.

## 0. 실행 순서

먼저 3단계 노트북을 끝까지 실행해 파라미터 아티팩트를 만든 뒤 실행한다.

```bash
uv run --project backend/notebooks --locked jupyter lab backend/notebooks/deal_model_phase4.ipynb
```

3단계의 최적 파라미터를 재사용하고 앙상블 결합만 반복 Group CV로 검증한다. TabICL은 GPU·MPS 메모리 충돌을 막기 위해 모든 Fold를 순차 실행한다.

In [1]:
import hashlib
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone, is_classifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

assert preprocessing_notebook.exists(), f"전처리 노트북이 없습니다: {preprocessing_notebook}"
ipython = get_ipython()
assert ipython is not None, "이 파일은 Jupyter에서 실행해야 합니다."
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = globals()["X_train_raw"]
y_train = np.asarray(globals()["y_train"], dtype=int)
train_group_ids = np.asarray(globals()["train_group_ids"])
train_mask_set_labels = X_train_raw.index.get_level_values("mask_set").to_numpy()
X_test_raw_sets = globals()["X_test_raw_sets"]
y_test = np.asarray(globals()["y_test"], dtype=int)
MODEL_FEATURE_NAMES = globals()["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = globals()["CATEGORY_VALUES"]

CLASSIFICATION_THRESHOLD = 0.5
OUTER_SEEDS = (1, 11, 21)
OUTER_FOLDS = 5
INNER_FOLDS = 3
GES_STEPS = 25
repo_root = preprocessing_notebook.parents[2]
PHASE3_SELECTION_PATH = (
    repo_root / "backend" / "pipeline" / "artifacts" / "deal_phase3_selection.joblib"
)
PHASE4_PREDICTION_PATH = (
    repo_root / "backend" / "pipeline" / "artifacts" / "deal_phase4_predictions.joblib"
)

print(f"Train: {X_train_raw.shape}, 동일 입력 그룹: {len(np.unique(train_group_ids))}")
print(f"평가: {OUTER_FOLDS}-Fold × {len(OUTER_SEEDS)}개 시드")
print(f"앙상블 내부 OOF: {INNER_FOLDS}-Fold, GES: {GES_STEPS}회")

데이터 전처리 검증을 통과했습니다.
Train: (3130, 13), 동일 입력 그룹: 138
평가: 5-Fold × 3개 시드
앙상블 내부 OOF: 3-Fold, GES: 25회


### 해석

- 독립 학습 단위는 마스킹 후 3,130행이 아니라 138개 입력 그룹이다.
- 한 번의 Fold 배정 운에 순위가 좌우되지 않도록 바깥 5-Fold를 시드 3개로 반복한다.
- 동일 원본과 마스킹 변형은 모든 바깥·안쪽 분할에서 같은 Fold에 유지된다.

## 1. 3단계 파라미터와 데이터 일치 확인

In [2]:
def training_data_signature() -> str:
    parts = (
        pd.util.hash_pandas_object(X_train_raw.astype("string"), index=True)
        .to_numpy(dtype=np.uint64)
        .tobytes(),
        np.asarray(y_train, dtype=np.int8).tobytes(),
        np.asarray(train_group_ids, dtype=np.uint64).tobytes(),
    )
    return hashlib.sha256(b"".join(parts)).hexdigest()


assert PHASE3_SELECTION_PATH.exists(), (
    "3단계 파라미터 아티팩트가 없습니다. deal_model_phase3.ipynb를 끝까지 실행하세요: "
    f"{PHASE3_SELECTION_PATH}"
)
selection_artifact = joblib.load(PHASE3_SELECTION_PATH)
assert selection_artifact["schema_version"] == 2
assert selection_artifact["data_signature"] == training_data_signature(), (
    "3단계와 현재 전처리 데이터가 다릅니다. 3단계를 현재 데이터로 다시 실행하세요."
)
assert selection_artifact["model_feature_names"] == list(MODEL_FEATURE_NAMES)
best_params = selection_artifact["best_params"]
display(
    pd.DataFrame(
        [
            {
                "model": model_name,
                "phase3_cv_brier": selection_artifact["cv_brier"][model_name],
                "best_params": params,
            }
            for model_name, params in best_params.items()
        ]
    ).sort_values("phase3_cv_brier", ignore_index=True)
)

,model,phase3_cv_brier,best_params
0,CatBoost,0.194832,"{'random_strength': 0.5, 'learning_rate': 0.01..."
1,LogisticRegression,0.197362,"{'classifier__C': 0.03, 'classifier__class_wei..."
2,MultinomialNB,0.198054,"{'classifier__alpha': 75.0, 'classifier__fit_p..."
3,ExtraTrees,0.198232,"{'classifier__max_depth': 8, 'classifier__max_..."
4,TabICL,0.213916,"{'average_logits': True, 'feat_shuffle_method'..."


### 해석

3단계와 4단계의 데이터 내용·행 순서·그룹 ID가 모두 같을 때만 앙상블을 진행한다. 4단계의 CV 수치는 이 일치된 데이터에서 선택된 3단계 최적 파라미터를 고정한 조건부 비교값으로 해석한다.

## 2. 모델별 입력 처리를 포함한 최종 기본 모델 구성

In [3]:
def make_one_hot_model(classifier):
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


if torch.backends.mps.is_available():
    tabicl_device = "mps"
elif torch.cuda.is_available():
    tabicl_device = "cuda"
else:
    tabicl_device = "cpu"

RANDOM_STATE = OUTER_SEEDS[0]
base_models = {
    "LogisticRegression": make_one_hot_model(
        LogisticRegression(max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE)
    ),
    "MultinomialNB": make_one_hot_model(MultinomialNB()),
    "ExtraTrees": make_one_hot_model(ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1)),
    "CatBoost": CatBoostClassifier(
        cat_features=tuple(MODEL_FEATURE_NAMES),
        loss_function="Logloss",
        verbose=False,
        allow_writing_files=False,
        random_seed=RANDOM_STATE,
        thread_count=1,
    ),
    "TabICL": TabICLClassifier(
        batch_size=8,
        kv_cache=False,
        allow_auto_download=True,
        device=tabicl_device,
        use_fa3="auto",
        offload_mode="auto",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=False,
    ),
}
for model_name, model in base_models.items():
    model.set_params(**best_params[model_name])
    assert is_classifier(model)
    assert hasattr(model, "predict_proba")

base_model_names = list(base_models)
model_inputs = {
    "LogisticRegression": "모델 내부 원핫 39개",
    "MultinomialNB": "모델 내부 원핫 39개",
    "ExtraTrees": "모델 내부 원핫 39개",
    "CatBoost": "원본 범주형 13개",
    "TabICL": "원본 범주형 13개",
}
display(
    pd.DataFrame(
        [
            {"model": name, "input": model_inputs[name], "parameters": model.get_params()}
            for name, model in base_models.items()
        ]
    )
)

,model,input,parameters
0,LogisticRegression,모델 내부 원핫 39개,"{'memory': None, 'steps': [('onehot', OneHotEn..."
1,MultinomialNB,모델 내부 원핫 39개,"{'memory': None, 'steps': [('onehot', OneHotEn..."
2,ExtraTrees,모델 내부 원핫 39개,"{'memory': None, 'steps': [('onehot', OneHotEn..."
3,CatBoost,원본 범주형 13개,"{'loss_function': 'Logloss', 'thread_count': 1..."
4,TabICL,원본 범주형 13개,"{'allow_auto_download': True, 'average_logits'..."


### 해석

- 모든 모델은 같은 원본 13개 컬럼을 받지만 LR·NB·ExtraTrees만 자신의 파이프라인에서 원핫으로 변환한다.
- CatBoost·TabICL은 3단계에서 선택된 원본 범주형 파라미터를 그대로 사용한다.
- 앙상블은 서로 다른 전처리를 강제로 통일하지 않고 각 모델의 Won 확률만 결합한다.
- 3단계에서 충분히 탐색한 최적 파라미터를 고정해, 4단계에서는 같은 튜닝을 반복하지 않고 앙상블 효과만 검증한다.

## 3. 공통 지표와 GES 함수

In [4]:
METRIC_NAMES = (
    "brier",
    "logloss",
    "auc",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "fpr",
    "tn",
    "fp",
    "fn",
    "tp",
)


def positive_class_probability(estimator, X):
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


def calculate_metrics(y_true, probability):
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp)
    return {
        "brier": brier_score_loss(y_true, probability),
        "logloss": log_loss(y_true, probability, labels=[0, 1]),
        "auc": roc_auc_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "specificity": specificity,
        "fpr": 1 - specificity,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def calculate_mask_set_averaged_metrics(y_true, probability, mask_set_labels):
    """Train의 10개 마스킹 세트를 각각 평가한 뒤 Test와 같은 방식으로 평균한다."""
    set_metrics = pd.DataFrame(
        [
            calculate_metrics(
                y_true[mask_set_labels == set_name],
                probability[mask_set_labels == set_name],
            )
            for set_name in np.unique(mask_set_labels)
        ]
    )
    return set_metrics.mean().to_dict()


def greedy_ensemble_weights(y_true, probability_matrix, steps=GES_STEPS):
    """OOF Brier를 가장 많이 낮추는 모델을 반복 선택해 비음수 가중치를 만든다."""
    counts = np.zeros(probability_matrix.shape[1], dtype=int)
    probability_sum = np.zeros(len(y_true), dtype=float)
    history = []
    for step in range(steps):
        candidate_brier = [
            brier_score_loss(y_true, (probability_sum + probability_matrix[:, index]) / (step + 1))
            for index in range(probability_matrix.shape[1])
        ]
        selected_index = int(np.argmin(candidate_brier))
        counts[selected_index] += 1
        probability_sum += probability_matrix[:, selected_index]
        history.append(
            {
                "step": step + 1,
                "selected_model": base_model_names[selected_index],
                "brier": candidate_brier[selected_index],
            }
        )
    return counts / counts.sum(), pd.DataFrame(history)


print("공통 지표와 GES 함수를 준비했습니다.")

공통 지표와 GES 함수를 준비했습니다.


### 해석

GES는 같은 모델을 여러 번 선택할 수 있다. 25회 중 선택된 횟수의 비율이 최종 가중치가 되며 선택되지 않은 모델은 자동으로 0이 된다.

## 4. 반복 Group CV

In [5]:
repeat_base_oof = []
repeat_ges_oof = []
repeat_stacking_oof = []
fold_weight_rows = []

for repeat_number, outer_seed in enumerate(OUTER_SEEDS, start=1):
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=outer_seed)
    outer_splits = list(outer_cv.split(X_train_raw, y_train, groups=train_group_ids))
    base_oof = np.full((len(y_train), len(base_models)), np.nan, dtype=float)
    ges_oof = np.full(len(y_train), np.nan, dtype=float)
    stacking_oof = np.full(len(y_train), np.nan, dtype=float)

    for fold_number, (outer_train, outer_valid) in enumerate(outer_splits, start=1):
        assert set(train_group_ids[outer_train]).isdisjoint(set(train_group_ids[outer_valid]))
        X_outer_train = X_train_raw.iloc[outer_train]
        y_outer_train = y_train[outer_train]
        groups_outer_train = train_group_ids[outer_train]

        inner_cv = StratifiedGroupKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=outer_seed + fold_number,
        )
        inner_splits = list(inner_cv.split(X_outer_train, y_outer_train, groups=groups_outer_train))
        for inner_train, inner_valid in inner_splits:
            assert set(groups_outer_train[inner_train]).isdisjoint(
                set(groups_outer_train[inner_valid])
            )

        inner_oof_matrix = np.full((len(outer_train), len(base_models)), np.nan, dtype=float)
        outer_valid_matrix = np.full((len(outer_valid), len(base_models)), np.nan, dtype=float)

        for model_index, (model_name, model) in enumerate(base_models.items()):
            # 기본 모델 파라미터는 3단계 최적값으로 고정한다. 안쪽 CV는 앙상블
            # 가중치와 Stacking 학습에 필요한 OOF 확률을 만드는 데만 사용한다.
            model_search_jobs = 1 if model_name == "TabICL" else -1
            inner_probability_matrix = cross_val_predict(
                clone(model),
                X_outer_train,
                y_outer_train,
                groups=groups_outer_train,
                cv=inner_splits,
                n_jobs=model_search_jobs,
                method="predict_proba",
            )
            inner_oof_matrix[:, model_index] = inner_probability_matrix[:, 1]

            fitted_outer_model = clone(model).fit(X_outer_train, y_outer_train)
            outer_valid_matrix[:, model_index] = positive_class_probability(
                fitted_outer_model, X_train_raw.iloc[outer_valid]
            )

        assert np.isfinite(inner_oof_matrix).all()
        assert np.isfinite(outer_valid_matrix).all()
        base_oof[outer_valid] = outer_valid_matrix

        fold_weights, _ = greedy_ensemble_weights(y_outer_train, inner_oof_matrix)
        ges_oof[outer_valid] = outer_valid_matrix @ fold_weights

        stacking_model = LogisticRegression(
            C=0.1, max_iter=3000, solver="lbfgs", random_state=outer_seed
        ).fit(inner_oof_matrix, y_outer_train)
        stacking_oof[outer_valid] = positive_class_probability(stacking_model, outer_valid_matrix)

        fold_weight_rows.append(
            {
                "repeat": repeat_number,
                "outer_seed": outer_seed,
                "fold": fold_number,
                **dict(zip(base_model_names, fold_weights, strict=True)),
            }
        )
        print(f"반복 {repeat_number}/{len(OUTER_SEEDS)}, Fold {fold_number}/{OUTER_FOLDS} 완료")

    assert np.isfinite(base_oof).all()
    assert np.isfinite(ges_oof).all()
    assert np.isfinite(stacking_oof).all()
    repeat_base_oof.append(base_oof)
    repeat_ges_oof.append(ges_oof)
    repeat_stacking_oof.append(stacking_oof)

repeat_base_oof = np.stack(repeat_base_oof)
repeat_ges_oof = np.stack(repeat_ges_oof)
repeat_stacking_oof = np.stack(repeat_stacking_oof)
fold_ges_weights = pd.DataFrame(fold_weight_rows)
display(fold_ges_weights.round(3))

반복 1/3, Fold 1/5 완료


반복 1/3, Fold 2/5 완료


반복 1/3, Fold 3/5 완료


반복 1/3, Fold 4/5 완료


반복 1/3, Fold 5/5 완료


...

반복 3/3, Fold 2/5 완료


반복 3/3, Fold 3/5 완료


반복 3/3, Fold 4/5 완료


반복 3/3, Fold 5/5 완료


,repeat,outer_seed,fold,LogisticRegression,MultinomialNB,ExtraTrees,CatBoost,TabICL
0,1,1,1,0.00,0.48,0.08,0.12,0.32
1,1,1,2,0.00,0.28,0.36,0.04,0.32
2,1,1,3,0.00,0.16,0.32,0.40,0.12
3,1,1,4,0.00,0.36,0.00,0.64,0.00
4,1,1,5,0.00,0.28,0.04,0.60,0.08
5,2,11,1,0.00,0.40,0.00,0.36,0.24
6,2,11,2,0.00,0.36,0.00,0.64,0.00
7,2,11,3,0.00,0.60,0.00,0.24,0.16
8,2,11,4,0.00,0.20,0.00,0.60,0.20
9,2,11,5,0.24,0.16,0.00,0.44,0.16


### 해석

- 각 바깥 Validation 예측은 해당 행을 보지 않고 바깥 Train만 학습한 기본 모델에서 나온다.
- GES 가중치와 Stacking 메타모델도 바깥 Train 안의 3-Fold OOF 확률만 보고 학습한다.
- 하이퍼파라미터 안정성은 3단계에서 확인하고, 이 단계에서는 단일 모델 대비 앙상블의 개선과 반복 간 안정성을 확인한다.
- Fold별 GES 가중치가 크게 달라지면 특정 앙상블 구성이 안정적이지 않다는 뜻이다.

## 5. 전체 Train용 앙상블과 Test 확률 생성

In [6]:
# 반복 OOF 예측을 행별로 평균해 전체 Train에서 최종 가중치와 메타모델을 학습한다.
mean_base_oof = repeat_base_oof.mean(axis=0)
final_ges_weights, final_ges_history = greedy_ensemble_weights(y_train, mean_base_oof)
final_stacking_model = LogisticRegression(
    C=0.1, max_iter=3000, solver="lbfgs", random_state=OUTER_SEEDS[0]
).fit(mean_base_oof, y_train)

# 기본 모델은 전체 Train에 한 번씩만 다시 학습하고 모든 Test 세트의 확률을 보관한다.
fitted_base_models = {}
test_base_probability = {
    set_name: np.full((len(y_test), len(base_models)), np.nan, dtype=float)
    for set_name in X_test_raw_sets
}
for model_index, (model_name, model) in enumerate(base_models.items()):
    fitted_model = clone(model).fit(X_train_raw, y_train)
    fitted_base_models[model_name] = fitted_model
    for set_name, X_test in X_test_raw_sets.items():
        test_base_probability[set_name][:, model_index] = positive_class_probability(
            fitted_model, X_test
        )

final_weight_table = pd.DataFrame(
    {
        "model": base_model_names,
        "ges_weight": final_ges_weights,
        "selected_count": (final_ges_weights * GES_STEPS).round().astype(int),
    }
).sort_values("ges_weight", ascending=False, ignore_index=True)
display(final_weight_table.round(4))
display(final_ges_history.tail(10).round(6))

,model,ges_weight,selected_count
0,CatBoost,0.64,16
1,MultinomialNB,0.36,9
2,LogisticRegression,0.00,0
3,ExtraTrees,0.00,0
4,TabICL,0.00,0


,step,selected_model,brier
15,16,CatBoost,0.192275
16,17,CatBoost,0.192281
17,18,MultinomialNB,0.192277
18,19,CatBoost,0.192276
19,20,MultinomialNB,0.192282
20,21,CatBoost,0.192276
21,22,CatBoost,0.192277
22,23,MultinomialNB,0.192278
23,24,CatBoost,0.192275
24,25,CatBoost,0.192278


### 해석

- 동일 가중 Voting은 다섯 확률의 단순 평균이다.
- GES는 반복 OOF 평균에서 Brier를 줄인 모델만 남겨 최종 가중치를 만든다.
- Stacking은 같은 OOF 확률을 입력으로 사용하되 규제된 LogisticRegression이 결합 경계를 학습한다.

## 6. 단일 모델과 앙상블 최종 비교

In [7]:
candidate_cv_probabilities = {}
for model_index, model_name in enumerate(base_model_names):
    candidate_cv_probabilities[model_name] = repeat_base_oof[:, :, model_index]
candidate_cv_probabilities["SoftVoting_All"] = repeat_base_oof.mean(axis=2)
candidate_cv_probabilities["GES_25"] = repeat_ges_oof
candidate_cv_probabilities["Stacking_LR"] = repeat_stacking_oof

candidate_test_probabilities = {
    model_name: {
        set_name: probability_matrix[:, model_index]
        for set_name, probability_matrix in test_base_probability.items()
    }
    for model_index, model_name in enumerate(base_model_names)
}
candidate_test_probabilities["SoftVoting_All"] = {
    set_name: probability_matrix.mean(axis=1)
    for set_name, probability_matrix in test_base_probability.items()
}
candidate_test_probabilities["GES_25"] = {
    set_name: probability_matrix @ final_ges_weights
    for set_name, probability_matrix in test_base_probability.items()
}
candidate_test_probabilities["Stacking_LR"] = {
    set_name: positive_class_probability(final_stacking_model, probability_matrix)
    for set_name, probability_matrix in test_base_probability.items()
}

result_rows = []
test_results_by_candidate = {}
for candidate_name, repeated_probability in candidate_cv_probabilities.items():
    repeat_metrics = pd.DataFrame(
        [
            calculate_mask_set_averaged_metrics(
                y_train,
                probability,
                train_mask_set_labels,
            )
            for probability in repeated_probability
        ]
    )
    test_results = pd.DataFrame(
        [
            {
                "test_set": set_name,
                **calculate_metrics(y_test, probability),
            }
            for set_name, probability in candidate_test_probabilities[candidate_name].items()
        ]
    )
    test_results_by_candidate[candidate_name] = test_results
    test_summary = test_results[list(METRIC_NAMES)].agg(["mean", "std"]).T
    result_rows.append(
        {
            "model": candidate_name,
            "kind": "ensemble"
            if candidate_name in {"SoftVoting_All", "GES_25", "Stacking_LR"}
            else "single",
            "cv_brier_mean": repeat_metrics["brier"].mean(),
            "cv_brier_std": repeat_metrics["brier"].std(ddof=0),
            "cv_logloss_mean": repeat_metrics["logloss"].mean(),
            "cv_auc_mean": repeat_metrics["auc"].mean(),
            "cv_accuracy_mean": repeat_metrics["accuracy"].mean(),
            "cv_precision_mean": repeat_metrics["precision"].mean(),
            "cv_recall_mean": repeat_metrics["recall"].mean(),
            "cv_f1_mean": repeat_metrics["f1"].mean(),
            "cv_fp_mean": repeat_metrics["fp"].mean(),
            "cv_fn_mean": repeat_metrics["fn"].mean(),
            "test_brier_mean": test_summary.loc["brier", "mean"],
            "test_brier_std": test_summary.loc["brier", "std"],
            "test_auc_mean": test_summary.loc["auc", "mean"],
            "test_accuracy_mean": test_summary.loc["accuracy", "mean"],
            "test_precision_mean": test_summary.loc["precision", "mean"],
            "test_recall_mean": test_summary.loc["recall", "mean"],
            "test_f1_mean": test_summary.loc["f1", "mean"],
            "test_fp_mean": test_summary.loc["fp", "mean"],
            "test_fn_mean": test_summary.loc["fn", "mean"],
        }
    )

comparison = pd.DataFrame(result_rows).sort_values(
    ["cv_brier_mean", "cv_auc_mean", "cv_accuracy_mean"],
    ascending=[True, False, False],
    ignore_index=True,
)
ensemble_comparison = comparison.loc[comparison["kind"] == "ensemble"].reset_index(drop=True)
display(comparison.round(6))
print(f"반복 CV Brier 기준 전체 1순위: {comparison.iloc[0]['model']}")
print(f"반복 CV Brier 기준 앙상블 1순위: {ensemble_comparison.iloc[0]['model']}")

,model,kind,cv_brier_mean,cv_brier_std,cv_logloss_mean,cv_auc_mean,cv_accuracy_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_fp_mean,cv_fn_mean,test_brier_mean,test_brier_std,test_auc_mean,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_fp_mean,test_fn_mean
0,Stacking_LR,ensemble,0.194406,0.001928,0.575930,0.766326,0.720873,0.695164,0.802935,0.745072,56.033333,31.333333,0.179131,0.005902,0.817164,0.734074,0.720534,0.772059,0.745083,20.4,15.5
1,SoftVoting_All,ensemble,0.194633,0.001798,0.577510,0.767950,0.722577,0.693498,0.813836,0.748789,57.233333,29.600000,0.177875,0.005611,0.819557,0.740741,0.720975,0.792647,0.754755,20.9,14.1
2,CatBoost,single,0.194730,0.002340,0.578848,0.766774,0.723855,0.695429,0.812369,0.749255,56.600000,29.833333,0.186809,0.006452,0.790222,0.740741,0.702151,0.844118,0.766377,24.4,10.6
3,GES_25,ensemble,0.195123,0.002515,0.579432,0.766453,0.720660,0.688315,0.823270,0.749682,59.333333,28.100000,0.180980,0.006676,0.806212,0.740741,0.708769,0.825000,0.762161,23.1,11.9
4,LogisticRegression,single,0.196919,0.001375,0.581281,0.762328,0.720021,0.696531,0.795807,0.742750,55.166667,32.466667,0.185205,0.006143,0.806080,0.713333,0.737183,0.673529,0.703239,16.5,22.2
5,MultinomialNB,single,0.196940,0.001893,0.582130,0.757841,0.715868,0.682415,0.825367,0.746959,61.166667,27.766667,0.181453,0.007900,0.800373,0.721481,0.720401,0.732353,0.725737,19.4,18.2
6,ExtraTrees,single,0.197144,0.002065,0.582248,0.762715,0.722258,0.692655,0.815094,0.748857,57.533333,29.400000,0.181914,0.005569,0.812028,0.742963,0.705081,0.842647,0.767612,24.0,10.7
7,TabICL,single,0.210237,0.002555,0.666228,0.742070,0.701704,0.683655,0.769602,0.723962,56.733333,36.633333,0.177992,0.005323,0.806629,0.730370,0.714184,0.776471,0.742526,21.2,15.2


반복 CV Brier 기준 전체 1순위: Stacking_LR
반복 CV Brier 기준 앙상블 1순위: Stacking_LR


### 해석 기준

- Train과 Test 모두 마스킹 10세트를 각각 평가한 뒤 평균하므로 AUC·Precision·F1도 같은 정의로 비교한다.
- `cv_*_mean`은 같은 모델을 세 가지 Fold 시드에서 평가한 평균이다. `cv_brier_std`가 크면 순위가 분할에 민감하다.
- Test 10세트의 표준편차는 신뢰구간이 아니라 어떤 4개 컬럼이 Unknown이 되는지에 따른 민감도다.
- Brier가 거의 같다면 AUC·Accuracy·Precision·Recall·FP·FN과 GES 가중치 안정성을 함께 보고, 운영 복잡도가 낮은 후보를 우선한다.
- Test 수치에 맞춰 가중치나 기본 모델 파라미터를 다시 바꾸지 않는다.

## 7. 기본 모델 확률의 다양성 확인

In [8]:
oof_probability_correlation = pd.DataFrame(
    mean_base_oof,
    columns=base_model_names,
).corr()
display(oof_probability_correlation.round(3))

expected_candidates = set(base_model_names) | {"SoftVoting_All", "GES_25", "Stacking_LR"}
assert set(candidate_cv_probabilities) == expected_candidates
assert set(candidate_test_probabilities) == expected_candidates
assert comparison["model"].nunique() == len(expected_candidates)
assert comparison.drop(columns=["model", "kind"]).notna().all().all()
assert np.allclose(final_ges_weights.sum(), 1.0)
assert (final_ges_weights >= 0).all()
assert all(
    len(results) == 10 and set(results["test_set"]) == set(X_test_raw_sets)
    for results in test_results_by_candidate.values()
)
assert all(
    np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    for repeated_probability in candidate_cv_probabilities.values()
    for probability in repeated_probability
)

phase4_prediction_artifact = {
    "schema_version": 1,
    "data_signature": training_data_signature(),
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "selected_candidate": "Stacking_LR",
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "cv_probabilities": repeat_stacking_oof,
    "cv_target": y_train,
    "cv_mask_set_labels": train_mask_set_labels,
    "test_probabilities": candidate_test_probabilities["Stacking_LR"],
    "test_target": y_test,
}
PHASE4_PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(phase4_prediction_artifact, PHASE4_PREDICTION_PATH)

print("4단계 반복 Group CV 앙상블 비교 검증을 통과했습니다.")
print(f"임계값 비교용 확률 저장: {PHASE4_PREDICTION_PATH}")

,LogisticRegression,MultinomialNB,ExtraTrees,CatBoost,TabICL
LogisticRegression,1.000,0.970,0.965,0.948,0.896
MultinomialNB,0.970,1.000,0.946,0.911,0.837
ExtraTrees,0.965,0.946,1.000,0.966,0.929
CatBoost,0.948,0.911,0.966,1.000,0.933
TabICL,0.896,0.837,0.929,0.933,1.000


4단계 반복 Group CV 앙상블 비교 검증을 통과했습니다.
임계값 비교용 확률 저장: backend/pipeline/artifacts/deal_phase4_predictions.joblib


### 해석

- 상관계수가 1에 가까운 모델끼리는 비슷한 확률을 내므로 함께 넣어도 이득이 작을 수 있다.
- 단일 순위가 낮은 모델도 다른 모델과 상관이 낮고 GES에서 반복 선택되면 앙상블에 기여한다.
- 이 노트북은 최종 후보 비교까지 수행한다. 선택된 후보의 직렬화와 서비스 연결은 결과를 확인한 뒤 별도로 진행한다.